# Network properties: Virus, bacteria and hosts

## Summary

In this notebook, we will create networks that allow us to study the interactions between virus, bacteria and hosts. Specifically, we will create three kinds of networks:

1. A bipartite ´Bacteria-Host´ network. We will create this one in two different flavors: a graph and a habitat-multigraph.
2. A tripartite ´Bacteria-Host-Virus´ network. We will create also two different flavors here.
3. A ´Bacteria-Virus´ network. In this case, it won't need to be a bipartite network.



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
import igraph  as ig
import networkx as nx
habitat_palette = {
    "Crop": "#0CAB7B", 
    "Edge": "#3A8DFA", 
    "Oak": "#FB2231",
    "Wasteland": "#FFC51F"
}

from skbio.diversity.alpha import shannon, chao1, faith_pd
from skbio import TreeNode
import scikit_posthocs as sp
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from rdflib import Graph
plt.rcParams['svg.fonttype'] = 'none'

# Writing the corresponding networks from SPAQRL queries 

In [ ]:
data = Graph()
data.parse(open("data/miripvir.2025-07-09.ttl"))

In [ ]:
res = data.query(
"""
PREFIX mvrtaxon: <http://localhost:8000/taxon/>
PREFIX mvrlib: <http://localhost:8000/library/>
PREFIX mvront: <http://localhost:8000/ont/>
PREFIX mvrcol: <http://localhost:8000/collection/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 
PREFIX emi: <https://purl.org/emi#>
PREFIX uniprotrdfs: <http://purl.uniprot.org/core/>

SELECT  ?hit_taxid ?hit_scientific_name ?hit_kingdom ?host_taxid ?host_scientific_name ?habitat ?lib_label ?pab
WHERE {
    ?lib mvront:Reported ?hit .
    ?lib rdfs:label ?lib_label .
    ?hit uniprotrdfs:Taxon ?hit_taxon .
    ?lib mvront:Obtained_from ?host .
    ?host uniprotrdfs:Taxon ?host_taxon .
    ?hit_taxon dwc:taxonID ?hit_taxid .
    ?hit_taxon dwc:scientificName ?hit_scientific_name .
    ?hit_taxon uniprotrdfs:Kingdom ?hit_kingdom .
    ?host_taxon dwc:taxonID ?host_taxid .
    ?host_taxon dwc:scientificName ?host_scientific_name .
    ?lib mvront:Sampled_from ?s .
    ?s wlo:habitat ?habitat .
    OPTIONAL {
        ?hit_taxon mvront:is_pab ?pab .
    }
}

"""
)
tripartite = pd.DataFrame(res, columns=list(map(str, res.vars)))
tripartite['hit_scientific_name'] = tripartite['hit_scientific_name'].apply(str)
tripartite['host_scientific_name'] = tripartite['host_scientific_name'].apply(str)
tripartite['host_taxid'] = tripartite['host_taxid'].apply(int)
tripartite['hit_taxid'] = tripartite['hit_taxid'].apply(int)
tripartite['habitat'] = tripartite['habitat'].apply(str)
tripartite['hit_kingdom'] = tripartite['hit_kingdom'].apply(str)
# detections['number_hits'] = detections['number_hits'].astype(int) 
# detections['scientific_name'] = detections['scientific_name'].apply(lambda x: str(x))
# detections['gtdb_genome_representative'] = detections['gtdb_genome_representative'].apply(lambda x: str(x))
# detections['site'] = detections['site'].apply(lambda x: str(x))
# detections['habitat'] = detections['habitat'].apply(lambda x: str(x))
# detections
tripartite['pab'] = tripartite['pab'].apply(lambda x: False if pd.isna(x) else True)
tripartite['is_valid'] = tripartite.apply(lambda x: x.pab or x.hit_kingdom == 'Virus', axis=1)
tripartite

In [ ]:
tripartite = tripartite.query('is_valid == True').copy()

## Graph

In [ ]:
tripartite_counts_by_pair = tripartite.value_counts(
    ['hit_taxid', 'hit_scientific_name', 'hit_kingdom', 'host_taxid', 'host_scientific_name']
).reset_index()
tripartite_counts_by_pair

In [ ]:
G = nx.Graph()
for _, row in tripartite.drop_duplicates(subset=['hit_taxid', 'hit_scientific_name', 'hit_kingdom'], keep='first').iterrows():
    G.add_node(row.hit_taxid, scientific_name=row.hit_scientific_name, kingdom=row.hit_kingdom)

for _, row in tripartite.drop_duplicates(['host_taxid', 'host_scientific_name']).iterrows():
    G.add_node(row.host_taxid, scientific_name=row.host_scientific_name)

for _, row in tripartite_counts_by_pair.iterrows():
    G.add_edge(row.hit_taxid, row.host_taxid, weight=row['count'])

G

In [ ]:
nx.draw(G)

In [ ]:
bactvirus_host_adjacency = tripartite_counts_by_pair.pivot(index=['hit_taxid'], columns=['host_taxid'], values='count').fillna(0)
bactvirus_host_adjacency

In [ ]:
bactvirus_host_adjacency.to_csv(
    "raw/networks/virus-bacteria-host.adjacency.full.csv", sep=';'
)

## Multigraph

In [ ]:
tripartite_counts_by_pair_habitat = tripartite.value_counts(
    ['hit_taxid', 'hit_scientific_name', 'hit_kingdom', 'host_taxid', 'host_scientific_name', 'habitat']
).reset_index()
tripartite_counts_by_pair_habitat

In [ ]:
M = nx.MultiGraph()
for _, row in tripartite.drop_duplicates(subset=['hit_taxid', 'hit_scientific_name', 'hit_kingdom'], keep='first').iterrows():
    M.add_node(row.hit_taxid, scientific_name=row.hit_scientific_name, kingdom=row.hit_kingdom, role=row.hit_kingdom)

for _, row in tripartite.drop_duplicates(['host_taxid', 'host_scientific_name']).iterrows():
    M.add_node(row.host_taxid, scientific_name=row.host_scientific_name, role='host')

for _, row in tripartite_counts_by_pair_habitat.iterrows():
    M.add_edge(row.hit_taxid, row.host_taxid, weight=row['count'], habitat=row['habitat'])

M

In [ ]:
M.nodes(data=True)

In [ ]:
nx.draw(M)

In [ ]:
nx.write_graphml(M, "raw/networks/virus-bacteria-host.multigraph.full.csv")

# iGraph plot

In [ ]:
role_palette = {
    'host': "#73deac",
    'Virus': "#e58bb1",
    'Bacteria': "#8bc8e5",
}

g = ig.read("raw/networks/virus-bacteria-host.multigraph.full.csv", format='graphml')
edge_colors = [habitat_palette[edge['habitat']] for edge in g.es]
node_colors = [role_palette[vertex['role']] for vertex in g.vs]
degrees = g.degree()
min_degree = min(degrees)
max_degree = max(degrees)
normalized_sizes = [10 + 20 * (degree - min_degree) / (max_degree - min_degree) for degree in degrees]
g.summary()

In [ ]:
layout = g.layout("kk")
ig.plot(g, layout=layout, edge_color=edge_colors, vertex_color=node_colors, vertex_size=normalized_sizes)

In [ ]:
layout = g.layout("fr")
ig.plot(g, layout=layout, edge_color=edge_colors, vertex_color=node_colors, vertex_size=normalized_sizes, target="figures/virus-bacteria-host.multigraph.full.force-layout.svg")

# Virus-bacteria graph

## Graph

In [ ]:
u_virus = tripartite[['hit_taxid', 'lib_label', 'habitat', 'hit_kingdom']].query('hit_kingdom == "Virus"')
u_bacteria = tripartite[['hit_taxid', 'lib_label', 'habitat', 'hit_kingdom']].query('hit_kingdom == "Bacteria"')
virus_bacteria_interactions = pd.merge(u_virus, u_bacteria, on=['lib_label', 'habitat'], suffixes=['_virus', '_bacteria'])
virus_bacteria_interactions = virus_bacteria_interactions.value_counts(
    ['hit_taxid_virus', 'hit_taxid_bacteria', 'habitat']
).reset_index()
virus_bacteria_interactions

In [ ]:
virus_bacteria_adjacency = virus_bacteria_interactions.groupby(['hit_taxid_bacteria', 'hit_taxid_virus'], as_index=False)['count'].sum().pivot(
    index='hit_taxid_bacteria', columns='hit_taxid_virus', values='count'
).fillna(0).to_csv("raw/networks/virus-bacteria.full.adjacency.csv", sep=';')

## Multigraph

In [ ]:
u_virus = tripartite[['hit_taxid', 'lib_label', 'habitat', 'hit_kingdom']].query('hit_kingdom == "Virus"')
u_bacteria = tripartite[['hit_taxid', 'lib_label', 'habitat', 'hit_kingdom']].query('hit_kingdom == "Bacteria"')
virus_bacteria_interactions = pd.merge(u_virus, u_bacteria, on=['lib_label', 'habitat'], suffixes=['_virus', '_bacteria'])
virus_bacteria_interactions = virus_bacteria_interactions.value_counts(
    ['hit_taxid_virus', 'hit_taxid_bacteria', 'habitat']
).reset_index()
virus_bacteria_interactions

In [ ]:
M = nx.MultiGraph()
for _, row in virus_bacteria_interactions.iterrows():
    M.add_edge(row.hit_taxid_virus, row.hit_taxid_bacteria, weight=row['count'], habitat=row['habitat'])

for _, row in tripartite.drop_duplicates(subset=['hit_taxid', 'hit_scientific_name', 'hit_kingdom'], keep='first').iterrows():
    if row.hit_taxid in list(M.nodes):
        M.add_node(row.hit_taxid, scientific_name=row.hit_scientific_name, kingdom=row.hit_kingdom, role=row.hit_kingdom)


In [ ]:
nx.write_graphml(M, "raw/networks/virus-bacteria.multigraph.full.csv")

In [ ]:
role_palette = {
    'host': "#73deac",
    'Virus': "#e58bb1",
    'Bacteria': "#8bc8e5",
}

g = ig.read("raw/networks/virus-bacteria.multigraph.full.csv", format='graphml')
edge_colors = [habitat_palette[edge['habitat']] for edge in g.es]
node_colors = [role_palette[vertex['role']] for vertex in g.vs]
degrees = g.degree()
min_degree = min(degrees)
max_degree = max(degrees)
normalized_sizes = [10 + 20 * (degree - min_degree) / (max_degree - min_degree) for degree in degrees]
g.summary()

In [ ]:
layout = g.layout("fr")
ig.plot(g, layout=layout, edge_color=edge_colors, vertex_color=node_colors, vertex_size=normalized_sizes, target="figures/virus-bacteria.multigraph.full.force-layout.svg")

# Bacteria-Host

## Graph

In [ ]:
tripartite_counts_by_pair = tripartite.value_counts(
    ['hit_taxid', 'hit_scientific_name', 'hit_kingdom', 'host_taxid', 'host_scientific_name']
).reset_index().query('hit_kingdom == "Bacteria"')
tripartite_counts_by_pair

In [ ]:
bacteria_host_adjacency = tripartite_counts_by_pair.pivot(index=['hit_taxid'], columns=['host_taxid'], values='count').fillna(0)
bacteria_host_adjacency

In [ ]:
bacteria_host_adjacency.to_csv(
    "raw/networks/bacteria-host.adjacency.full.csv", sep=';'
)

## Multigraph

In [ ]:
tripartite_counts_by_pair_habitat = tripartite.value_counts(
    ['hit_taxid', 'hit_scientific_name', 'hit_kingdom', 'host_taxid', 'host_scientific_name', 'habitat']
).reset_index().query('hit_kingdom == "Bacteria"')
tripartite_counts_by_pair_habitat

In [ ]:
M = nx.MultiGraph()
for _, row in tripartite.drop_duplicates(subset=['hit_taxid', 'hit_scientific_name', 'hit_kingdom'], keep='first').query('hit_kingdom == "Bacteria"').iterrows():
    M.add_node(row.hit_taxid, scientific_name=row.hit_scientific_name, kingdom=row.hit_kingdom, role=row.hit_kingdom)

for _, row in tripartite.drop_duplicates(['host_taxid', 'host_scientific_name']).iterrows():
    if row.host_taxid in tripartite_counts_by_pair_habitat.host_taxid.tolist():
        M.add_node(row.host_taxid, scientific_name=row.host_scientific_name, role='host')

for _, row in tripartite_counts_by_pair_habitat.iterrows():
    M.add_edge(row.hit_taxid, row.host_taxid, weight=row['count'], habitat=row['habitat'])
M

In [ ]:
bactvirus_host_adjacency = tripartite_counts_by_pair.pivot(index=['hit_taxid'], columns=['host_taxid'], values='count').fillna(0)
bactvirus_host_adjacency

In [ ]:
bactvirus_host_adjacency.to_csv(
    "raw/networks/virus-bacteria-host.adjacency.full.csv", sep=';'
)

In [ ]:
nx.write_graphml(M, "raw/networks/bacteria-host.multigraph.full.csv")

In [ ]:
role_palette = {
    'host': "#73deac",
    'Virus': "#e58bb1",
    'Bacteria': "#8bc8e5",
}

g = ig.read("raw/networks/bacteria-host.multigraph.full.csv", format='graphml')
edge_colors = [habitat_palette[edge['habitat']] for edge in g.es]
node_colors = [role_palette[vertex['role']] for vertex in g.vs]
degrees = g.degree()
min_degree = min(degrees)
max_degree = max(degrees)
normalized_sizes = [10 + 20 * (degree - min_degree) / (max_degree - min_degree) for degree in degrees]
degree_threshold = 40
labels = [str(v['scientific_name']) if degree > degree_threshold else "" for v, degree in zip(g.vs, degrees)]
g.summary()

In [ ]:
layout = g.layout("fr")
ig.plot(g, layout=layout, edge_color=edge_colors, vertex_color=node_colors, vertex_size=normalized_sizes, target="figures/bacteria-host.multigraph.full.force-layout.svg", vertex_label=labels, vertex_label_size=5)

In [ ]:
tripartite_counts_by_pair_habitat.value_counts('host_scientific_name')

In [ ]:
tripartite_counts_by_pair_habitat.value_counts('hit_scientific_name')

### Network properties 

# Module enrichment

In [ ]:
modules = pd.read_csv("raw/networks/bacteria-host.modules.full.csv", sep='\s+').query('type == "col"')
modules['value'] = modules['value'].apply(lambda x: x[1:])
modules

In [ ]:
res = data.query(
"""
PREFIX mvrtaxon: <http://localhost:8000/taxon/>
PREFIX mvrlib: <http://localhost:8000/library/>
PREFIX mvront: <http://localhost:8000/ont/>
PREFIX mvrcol: <http://localhost:8000/collection/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 
PREFIX emi: <https://purl.org/emi#>
PREFIX uniprotrdfs: <http://purl.uniprot.org/core/>

SELECT ?host_taxid ?habitat
WHERE {
    
    ?lib mvront:Obtained_from ?host .
    ?host uniprotrdfs:Taxon ?host_taxon .
    ?host_taxon dwc:taxonID ?host_taxid .
    ?lib mvront:Sampled_from ?s .
    ?s wlo:habitat ?habitat .
}

"""
)
host_habitat = pd.DataFrame(res, columns=list(map(str, res.vars)))
host_habitat['host_taxid'] = host_habitat['host_taxid'].apply(str)
host_habitat['habitat'] = host_habitat['habitat'].apply(str)
host_habitat

In [ ]:
cont_table = pd.merge(
    modules, host_habitat, left_on='value', right_on='host_taxid'
).value_counts(
    ['index', 'habitat']
).reset_index().pivot(
    index='index', columns='habitat', values='count'
).fillna(0).astype(int)
cont_table

In [ ]:
contingency_test = stats.chi2_contingency(cont_table)
print(f"p-value = {contingency_test.pvalue:8.6e}")
print(f"stat    = {contingency_test.statistic:8.6e}")
print(f"dof     = {contingency_test.dof:8d}")

In [ ]:
contingency_test.pvalue